In [1]:
from skyfield.api import load, wgs84
import numpy as np
from numpy import array2string
from datetime import datetime, timedelta
import os
from scipy import io as sio
import pytz
import math

In [2]:
# TODO: define constants and set other parameters
c = 299792458.0 # speed of light in m/s
fc = 11.575e9 # carrier frequency in Hz
lo = 9.75e9 # local oscillator frequency in Hz
fs = 1e6 # sampling frequency in Hz
lam = c/fc # wavelength in m
d = lam/2 # antenna spacing in m
re = 6378e3 # earth radius in m
min_elevation = 60 # minimum elevation of visible sats

In [3]:
# Load tle from path

# change the path
# path = '/Users/ishani/UIUC/Research/starlink_ofdm/AoA_Sim/tles/10_31/STARLINK-6150.tle'
path = 'D:\\Desktop\\study\\2023_summer\\starlink_ofdm\\AoA_Sim\\tles\\10_31\\STARLINK-6150.tle'

satellites = load.tle_file(path)
print(satellites)

# % assume we have 4 synchronized LNBs, placed in the following way:
#  ------------         N
# | LNB2 LNB4 |       W + E
# | LNB1 LNB3 |         S
#  ------------
# initialize location of 4 LNBs with lat, lon = +40.11389690596162, -88.22484814695467 (UIUC)
lat_1, lon_1 = +40.11389690596162, -88.22484814695467
lnb_1 = wgs84.latlon(lat_1, lon_1)
lnb_2 = wgs84.latlon(lat_1+math.degrees(d/re), lon_1)
lnb_3 = wgs84.latlon(lat_1, lon_1+math.degrees(d/re)/math.cos(math.radians(lat_1)))
lnb_4 = wgs84.latlon(lat_1+math.degrees(d/re), lon_1+math.degrees(d/re)/math.cos(math.radians(lat_1)))
lnbs = [lnb_1, lnb_2, lnb_3, lnb_4]

# set start time and stop time, add 5 hours to account for time zone difference
ts = load.timescale()
start_time = datetime(2023, 10, 31, 0, 0, 0, tzinfo=pytz.UTC)+timedelta(hours=5) 
stop_time = datetime(2023, 10, 31, 0, 1, 0, tzinfo=pytz.UTC)+timedelta(hours=5)
# get start time as ts.utc
t = ts.utc(start_time)
n = np.arange(0, fs, 1)
i = 0


[<EarthSatellite STARLINK-6150 catalog #57119 epoch 2023-10-31 06:43:42 UTC>]


The following section tries to estimate the AoA (both azimuth & elevation) using MUSIC.

Assume that the satellite is transmitting at frequency $f_c+f=11.575 GHz+0.2 MHz$, i.e.,
$$y(t) = \exp(j2\pi(f+f_c)t),$$
where $f_c$ is the carrier frequency and $f$ is the narrowband tone's frequency.

Assume when it arrives at LNB-1 at UIUC, it becomes 
$$y_1(t) = A\exp(j2\pi(f+f_c)t)+n_1,$$
where $A$ is path loss, $n_1$ is the noise at LNB-1.

Similarly, for LNB-i (i=2,3,4), there will be time delays since the distance between LNB-i will be different from LNB-1, given by
$$y_i(t) = A\exp(j2\pi(f+f_c)\times (t-(r_i-r_1)/c))+n_i, \\
 = A\exp(-j2\pi(f+f_c)*(r_i-r_1)/c)\times \exp(j2\pi(f+f_c)t)+n_i, \\
 = A'\exp(j2\pi(f+f_c)t)+n_i,$$
where $r_i$ is the distance between the sat and the i-th LNB. If $r_i>r_1$ then it arrives later. Also assume that $A$ is same for different LNBs since they are very close to each other.

$y_i(t)$ is in anaglog domain, then LNB \& USRP down-converts it at sample it. In the most ideal case where we have synchronized LNBs and USRPs, the digital signal we get is removing $f_c$ and substituting continuous time $t$ with discrete time samples $n*T_s = n/F_s$, i.e.,
$$y_i[n] = A'\exp(j2\pi f n/f_s)+n_i,$$
from where we start our simulation

In [ ]:

while True:
    dist = (satellites[0]-gs).at(t)
    el, az, distance = dist.altaz()
    # print time, elevation, azimuth, latitude, longitude
    print(t.utc_iso(), el.degrees, az.degrees)
    _, _, the_range, _, _, range_rate = dist.frame_latlon_and_rates(gs)
    doppler_shift = fc*range_rate.km_per_s*1000/c
    
    # generate received signal TODO: double check
    received_sig_0 = np.exp(-1j*(2*np.pi*fc+doppler_shift)*(n[i]/fs+the_range.km/c)) # TODO: can comment this out
    # downocnvert to baseband
    received_sig_0 = received_sig_0*np.exp(-1j*(2*np.pi*doppler_shift)*(n[i]/fs+the_range.km/c))

    # TODO: replicate code for more antennas

    # TODO: get phase difference between antennas

    #TODO: get AoA

    # increment t which is ts.utc by 1 second
    t = ts.utc(t.utc_datetime()+timedelta(seconds=1))
    i += 1

    if t.utc_datetime() > stop_time:
        break
